# Basic FastHTML SSE Integration Example

> Demonstrates real-time tqdm progress tracking in FastHTML using Server-Sent Events (SSE) instead of HTMX polling. Shows how to capture tqdm output from background tasks, stream live progress updates via SSE, and handle task completion with automatic UI state management.

In [1]:
from fasthtml.common import *
import uuid, time

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.LIGHT)

# Add SSE extension script for HTMX (if using HTMX SSE features)
# Note: For this example, we'll use vanilla JavaScript EventSource instead

# Initialize monitor and runner
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Define a simple task that uses tqdm
def simple_task():
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(100), desc="Processing"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

In [5]:
# Create helper function for Progress bars
def create_progress_bar(value="0", max="100"):
    """Create a styled progress bar with consistent styling"""
    return Progress(
        value=str(value), 
        max=str(max), 
        cls=combine_classes(progress, progress_colors.primary, w.full)
    )

# Create helper function for Progress labels
def create_progress_label(text="Progress:"):
    """Create a styled progress label with consistent styling"""
    return P(text, cls=combine_classes(font_weight.bold))

# Create helper function for status messages
def create_status_message(text):
    """Create a styled status message with consistent styling"""
    return P(text, cls=combine_classes(m.t(2), font_size.sm))

# Create helper function for complete progress display
def create_progress_display(value="0", status_text="Waiting..."):
    """Create a complete progress display with label, bar, and status"""
    return Div(
        create_progress_label(),
        create_progress_bar(value=value, max="100"),
        create_status_message(status_text),
        id="progress-display"
    )

# Helper to render start button with appropriate state
def render_start_button(disabled=False):
    """Render start button with appropriate state"""
    btn_classes = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Button(
        "Start Task", 
        id="start-btn",
        hx_post="/start_task" if not disabled else None,
        hx_target="#progress-container",
        hx_swap="innerHTML",
        cls=btn_classes,
        disabled=disabled
    )

In [6]:
# Create global SSE manager script
sse_manager_script = """
window.sseManager = {
    currentSource: null,
    
    connect: function(jobId) {
        // Close any existing connection
        if (this.currentSource) {
            this.currentSource.close();
        }
        
        // Create new EventSource connection
        this.currentSource = new EventSource('/stream?job_id=' + jobId);
        
        // Handle incoming messages
        this.currentSource.onmessage = function(event) {
            const data = JSON.parse(event.data);
            
            // Update progress bar
            const progressBar = document.querySelector('#progress-display progress');
            if (progressBar) {
                progressBar.value = data.progress;
            }
            
            // Update status text
            const statusText = document.querySelector('#progress-display p:last-child');
            if (statusText) {
                if (data.bars && Object.keys(data.bars).length > 0) {
                    // Get first bar's info
                    const firstBar = Object.values(data.bars)[0];
                    statusText.textContent = `${firstBar.desc}: ${firstBar.pct.toFixed(1)}% (${firstBar.cur}/${firstBar.tot})`;
                } else {
                    statusText.textContent = `Processing... ${data.progress.toFixed(1)}%`;
                }
            }
            
            // Handle completion
            if (data.completed) {
                window.sseManager.currentSource.close();
                window.sseManager.currentSource = null;
                
                // Update UI to completed state
                if (statusText) {
                    statusText.textContent = 'Completed!';
                }
                
                // Re-enable the start button
                const startBtn = document.getElementById('start-btn');
                if (startBtn) {
                    startBtn.disabled = false;
                    startBtn.classList.remove('btn-disabled');
                    startBtn.setAttribute('hx-post', '/start_task');
                    htmx.process(startBtn);  // Re-process HTMX attributes
                }
            }
        };
        
        // Handle errors
        this.currentSource.onerror = function(error) {
            console.error('SSE Error:', error);
            window.sseManager.currentSource.close();
            window.sseManager.currentSource = null;
        };
    },
    
    disconnect: function() {
        if (this.currentSource) {
            this.currentSource.close();
            this.currentSource = null;
        }
    }
};
"""

In [7]:
@rt("/")
def index():
    return create_test_page(
        "Basic SSE Progress Demo",
        Div(
            H2("Simple Progress Tracking with SSE"),
            render_start_button(disabled=False),
            Div(
                P("Ready", cls=combine_classes(font_weight.bold, m.t(4))),
                id="progress-container",
                cls=str(m.t(4))
            ),
            cls=str(p(8))
        ),
        # Add the global SSE manager script
        Script(sse_manager_script)
    )

In [8]:
# API endpoints using SSE streaming
@rt("/start_task")
def start_task():
    job_id = str(uuid.uuid4())
    
    runner.start(
        job_id, 
        simple_task,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Return initial UI with SSE connection trigger
    return Div(
        # Disable button immediately
        Script("""
            const btn = document.getElementById('start-btn');
            if (btn) {
                btn.disabled = true;
                btn.classList.add('btn-disabled');
                btn.removeAttribute('hx-post');
            }
        """),
        # Progress display
        create_progress_display(
            value="0",
            status_text="Starting..."
        ),
        # Connect to SSE stream
        Script(f"window.sseManager.connect('{job_id}');")
    )

@rt("/stream")
def stream(job_id: str):
    """SSE endpoint that streams progress updates"""
    # Use FastHTML's EventStream with the sse_stream generator
    return EventStream(
        sse_stream(
            monitor, 
            job_id, 
            interval=0.1,      # Check for updates every 100ms
            heartbeat=10.0,    # Send keep-alive every 10 seconds
            wait_for_start=True,  # Wait for job to start
            start_timeout=5.0     # Wait up to 5 seconds for job to start
        )
    )

In [9]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [10]:
# Stop server when done
server.stop()